# Calidad de los datos, desbalance, preprocesamiento y partición

### Entorno y persistencia de resultados

En la celda siguiente fijé tres cosas que condicionan la reproducibilidad del experimento.

**Semilla fija.** Usé `RANDOM_STATE = 42` en la partición train/test y en el ajuste de todos los
modelos. Sin ella, cada ejecución produciría particiones distintas y las métricas no serían
comparables entre corridas ni verificables por un tercero.

**Persistencia en Google Drive.** Escribí los resultados en `MyDrive/bank_marketing` y no en el
disco temporal de Colab, que se borra al cerrar la sesión. El notebook 03 deja las particiones
preparadas y los notebooks 04 y 05 las consumen tal cual, de modo que todos operan sobre exactamente
los mismos datos. Si el montaje no se completa, la celda interrumpe la ejecución en lugar de escribir
en una ubicación volátil. Fuera de Colab no hace falta Drive, y los resultados se escriben en la
carpeta del repositorio (los gráficos en `splits/`).

**Registro de las figuras.** La función `guardar` escribe cada gráfico en `splits/` como PNG a 200
dpi. La invoco antes de `plt.show()`, porque mostrar la figura vacía el buffer de matplotlib y el
archivo saldría en blanco.

In [1]:
import os, io, time, json, zipfile, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedó montado. Vuelve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = "/content/drive/MyDrive/bank_marketing"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)

def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=200, bbox_inches="tight", facecolor="white")
    print("Gráfico guardado:", destino)

print("Guardando en Google Drive" if EN_DRIVE else "Google Drive no disponible: guardando localmente")
print("Ruta de trabajo:", RUTA)

Google Drive no disponible: guardando localmente
Ruta de trabajo: C:\Users\USUARIO\Documents\6_SEMESTRE\MACHINE LEARNING\Bank_Marketing\bank_marketing


In [2]:
URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
ARCHIVO_LOCAL = os.path.join(RUTA, "bank-full.csv")

def cargar_datos():
    if os.path.exists(ARCHIVO_LOCAL):
        return pd.read_csv(ARCHIVO_LOCAL, sep=";")
    with urllib.request.urlopen(URL, timeout=120) as respuesta:
        externo = zipfile.ZipFile(io.BytesIO(respuesta.read()))
    interno = zipfile.ZipFile(io.BytesIO(externo.read("bank.zip")))
    datos = pd.read_csv(interno.open("bank-full.csv"), sep=";")
    datos.to_csv(ARCHIVO_LOCAL, sep=";", index=False)
    return datos

df = cargar_datos()
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])

Filas: 45211 | Columnas: 17


## Calidad de los datos

Revisé cuatro cosas: nulos, duplicados, categorías `unknown` y valores atípicos. Antes de cualquier
decisión retiré `duration`, por la fuga de información explicada en el notebook 01.

In [3]:
df = df.drop(columns=["duration"])
df["contrato"] = (df["y"] == "yes").astype(int)

print("Nulos reales:", int(df.isna().sum().sum()))
duplicados = int(df.drop(columns="contrato").duplicated().sum())
print("Filas duplicadas (sin duration):", duplicados)

Nulos reales: 0
Filas duplicadas (sin duration): 16


**Lectura.** Sin `duration` hay 16 filas idénticas en todas las predictoras.

### Decisión sobre duplicados

El conjunto no tiene un identificador de cliente. Al retirar `duration` pueden quedar filas idénticas en
todas las demás columnas que corresponden a clientes distintos con perfil parecido y no a registros
repetidos, y eliminarlas cambiaría la tasa de contratación observada. Por eso medí el efecto antes de
decidir.

In [4]:
sin_duplicados = df.drop_duplicates(subset=df.columns.drop("contrato").tolist())
print("Filas antes:", len(df), "| después:", len(sin_duplicados))
print("Tasa de contratación antes:", round(100 * df["contrato"].mean(), 3), "%")
print("Tasa de contratación después:", round(100 * sin_duplicados["contrato"].mean(), 3), "%")

Filas antes: 45211 | después: 45195
Tasa de contratación antes: 11.698 %
Tasa de contratación después: 11.703 %


**Lectura.** Eliminar esas 16 filas cambia la tasa de contratación de 11,698 % a 11,703 %, un efecto
despreciable. Como no hay identificador de cliente no se puede saber si son registros repetidos o
clientes distintos con el mismo perfil, así que los conservé.

### Categorías `unknown` y atípicos

Conservé las categorías `unknown` como una categoría más. No son nulos sino un valor que el banco
registró cuando no tenía el dato, y que un cliente no haya informado su ocupación o su educación puede
ser información en sí misma. No las imputé, porque imputar con la moda inventaría un dato que no existe.

No eliminé ninguna fila por atípica. Los valores extremos de `balance`, `campaign` y `previous` son
reales (saldos grandes, clientes contactados muchas veces) y los modelos basados en árboles no se ven
afectados por ellos. Donde sí importan es en el SVM, porque un valor extremo estira la escala y
distorsiona las distancias del kernel, y por eso elegí el escalador del SVM más abajo con evidencia,
comparando tres alternativas.

### Variable derivada

`pdays` vale -1 cuando el cliente no fue contactado en una campaña anterior, y ese código convive con
valores reales de días. Agregué `contactado_antes`, un indicador explícito de ese caso, para que los
modelos no tengan que inferirlo de un número negativo.

También agregué dos variables de fecha. El notebook 02 mostró que la tasa de contratación cambia mucho
según el mes, y el día del mes por sí solo no tiene una forma clara. Con `mes_num` (el mes como número del
1 al 12) y `dia_del_anio` (una aproximación del día del año, igual a 31 por los meses transcurridos más el
día) un modelo de árboles puede aislar un período concreto de la campaña con una sola división, en lugar
de combinar el mes y el día en varias. Las dos se conocen antes de la llamada, porque son la fecha en que
se hace, así que no introducen fuga. El conjunto no trae el año, por eso la fecha es solo aproximada.

In [5]:
df["contactado_antes"] = (df["pdays"] != -1).astype(int)
MESES = {m: i for i, m in enumerate(["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], 1)}
df["mes_num"] = df["month"].map(MESES)
df["dia_del_anio"] = (df["mes_num"] - 1) * 31 + df["day"]
print(df["contactado_antes"].value_counts())
print("\nVariables de fecha (primeras filas):")
print(df[["month", "day", "mes_num", "dia_del_anio"]].head())

print("\nValores unknown por variable:")
categoricas = df.select_dtypes(exclude="number").columns.drop("y").tolist()
print((df[categoricas] == "unknown").sum().loc[lambda s: s > 0])

contactado_antes
0    36954
1     8257
Name: count, dtype: int64

Variables de fecha (primeras filas):


  month  day  mes_num  dia_del_anio
0   may    5        5           129
1   may    5        5           129
2   may    5        5           129
3   may    5        5           129
4   may    5        5           129

Valores unknown por variable:
job            288
education     1857
contact      13020
poutcome     36959
dtype: int64


**Lectura.** `contactado_antes` vale 1 para 8.257 clientes (18,3 %) y 0 para el resto. La variable
`pdays` conserva los días reales para los que sí fueron contactados. Las variables de fecha traducen el
mes y el día a números, por ejemplo el 5 de mayo queda como mes 5 y día 129 del año.

## Cuantificación del desbalance

Calculé la proporción de cada clase y qué obtendría un modelo trivial que siempre responde `no`. Ese
modelo es la línea base que cualquier clasificador real tiene que superar.

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

conteo = df["y"].value_counts()
print(conteo)
print("Razón no:yes =", round(conteo["no"] / conteo["yes"], 2), ": 1")

trivial = DummyClassifier(strategy="most_frequent").fit(df[["age"]], df["contrato"])
prediccion = trivial.predict(df[["age"]])
print("\nModelo trivial (siempre 'no')")
print("Accuracy:", round(accuracy_score(df["contrato"], prediccion), 4))
print("F1 de la clase positiva:", round(f1_score(df["contrato"], prediccion, zero_division=0), 4))

y
no     39922
yes     5289
Name: count, dtype: int64
Razón no:yes = 7.55 : 1

Modelo trivial (siempre 'no')
Accuracy: 0.883
F1 de la clase positiva: 0.0


**Lectura.** El modelo trivial tiene accuracy de 0,883 y un F1 de cero para la clase positiva. Ese es
el piso. Cualquier modelo que valga la pena tiene que superar el cero en F1 y el 0,5 en ROC-AUC, y su
accuracy puede quedar por debajo de 0,883 sin que eso sea un fracaso, porque estará intentando
encontrar contratadores y no solo acertar la clase mayoritaria.

### Estrategia frente al desbalance

Apliqué tres medidas combinadas. Hice la partición **estratificada**, para que train y test conserven la
proporción real de la clase positiva. Usé **ponderación de clases** (`class_weight="balanced"` en SVM,
árbol y Random Forest, y `scale_pos_weight` en XGBoost), que hace que equivocarse con un contratador pese
tanto como el desbalance. Y tomé las decisiones de ajuste con **F1 de la clase positiva** y no con
accuracy. En el notebook 04 contrasté empíricamente el efecto de la ponderación.

## Partición estratificada

Separé primero y transformé después. Con esa secuencia es imposible que el escalador o el codificador
vean datos de prueba, por descuido o por diseño. Dejé el 80 % para entrenamiento y el 20 % para prueba,
con semilla fija.

In [7]:
from sklearn.model_selection import train_test_split

y = df["contrato"]
X = df.drop(columns=["y", "contrato"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Tasa de contratación en train:", round(100 * y_train.mean(), 3), "%")
print("Tasa de contratación en test: ", round(100 * y_test.mean(), 3), "%")

cols_cat = X.select_dtypes(exclude="number").columns.tolist()
cols_num = X.select_dtypes(include="number").columns.tolist()
print("\nNuméricas:", cols_num)
print("Categóricas:", cols_cat)

Train: (36168, 18) | Test: (9043, 18)
Tasa de contratación en train: 11.698 %
Tasa de contratación en test:  11.7 %

Numéricas: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous', 'contactado_antes', 'mes_num', 'dia_del_anio']
Categóricas: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


**Lectura.** La partición deja 36.168 clientes para entrenamiento y 9.043 para prueba, con una tasa de
contratación de 11,698 % y 11,70 % respectivamente. La estratificación funcionó. Hay 9 variables
numéricas (las 6 originales más `contactado_antes`, `mes_num` y `dia_del_anio`) y 9 categóricas.

## Preprocesamiento diferenciado

No todos los modelos necesitan lo mismo, así que diferencié el tratamiento.

**SVM.** El kernel RBF mide distancias entre clientes, de modo que una variable con rango grande
(`balance` llega a decenas de miles) dominaría a otra con rango chico (`previous`). Por eso escalé las
numéricas y codifiqué las categóricas con one-hot.

**Árbol, Random Forest y XGBoost.** Cada división compara una variable con un umbral, y ese orden no
cambia aunque la variable se multiplique o se desplace. No escalé nada y solo codifiqué las categóricas.

Para demostrar que escalar no afecta a un árbol, en la celda siguiente entrené el mismo árbol con y sin
escalado y las predicciones resultaron idénticas.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

def codificador():
    return OneHotEncoder(handle_unknown="ignore", drop="if_binary")

arbol_sin_escalar = Pipeline([
    ("prep", ColumnTransformer([("num", "passthrough", cols_num), ("cat", codificador(), cols_cat)])),
    ("clf", DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE))])
arbol_escalado = Pipeline([
    ("prep", ColumnTransformer([("num", StandardScaler(), cols_num), ("cat", codificador(), cols_cat)])),
    ("clf", DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE))])

pred_a = arbol_sin_escalar.fit(X_train, y_train).predict(X_test)
pred_b = arbol_escalado.fit(X_train, y_train).predict(X_test)
print("Predicciones idénticas con y sin escalado:", bool((pred_a == pred_b).all()))
print("F1 sin escalar:", round(f1_score(y_test, pred_a), 4), "| F1 escalado:", round(f1_score(y_test, pred_b), 4))

Predicciones idénticas con y sin escalado: True
F1 sin escalar: 0.3105 | F1 escalado: 0.3105


**Lectura.** Las predicciones del árbol son idénticas con y sin escalado, y el F1 es el mismo (0,311).
Eso verifica en la práctica que los modelos de árboles no necesitan escalar las variables y que se
puede ahorrar ese paso.

### Elección del escalador del SVM

Las variables `balance`, `campaign`, `previous` y `pdays` tienen colas largas, y el escalador estándar es
sensible a ellas porque usa media y desviación. Comparé tres alternativas sobre una submuestra
estratificada del train (el SVM es lento y la comparación solo necesita una referencia relativa):
`StandardScaler`, `RobustScaler` (mediana y rango intercuartílico) y `QuantileTransformer` (mapea cada
variable a una distribución normal por cuantiles). Las evalué con validación cruzada de 3 particiones
sobre F1 de la clase positiva y ROC-AUC.

In [9]:
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from sklearn.model_selection import StratifiedKFold, cross_validate

submuestra, _, y_sub, _ = train_test_split(
    X_train, y_train, train_size=6000, stratify=y_train, random_state=RANDOM_STATE)

escaladores = {
    "estandar": StandardScaler(),
    "robusto": RobustScaler(),
    "cuantil": QuantileTransformer(output_distribution="normal", n_quantiles=500, random_state=RANDOM_STATE),
}
validacion = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

filas = []
for nombre, escalador in escaladores.items():
    modelo = Pipeline([
        ("prep", ColumnTransformer([("num", escalador, cols_num), ("cat", codificador(), cols_cat)])),
        ("clf", SVC(kernel="rbf", C=1.0, class_weight="balanced", random_state=RANDOM_STATE))])
    r = cross_validate(modelo, submuestra, y_sub, cv=validacion, scoring=["f1", "roc_auc"], n_jobs=-1)
    filas.append({"escalador": nombre,
                  "f1_medio": r["test_f1"].mean(), "f1_desv": r["test_f1"].std(),
                  "auc_medio": r["test_roc_auc"].mean(), "auc_desv": r["test_roc_auc"].std()})

comparacion_escaladores = pd.DataFrame(filas).set_index("escalador").round(4)
print(comparacion_escaladores)

escalador_elegido = comparacion_escaladores["f1_medio"].idxmax()
print("\nEscalador elegido para el SVM:", escalador_elegido)

           f1_medio  f1_desv  auc_medio  auc_desv
escalador                                        
estandar     0.3859   0.0203     0.7413    0.0135
robusto      0.2843   0.0037     0.6971    0.0053
cuantil      0.3402   0.0166     0.7283    0.0070

Escalador elegido para el SVM: estandar


**Lectura.** En el SVM el escalador sí importa. El estándar obtiene un F1 de 0,386 y un AUC de 0,741,
claramente por encima del robusto (0,284 y 0,697) y del cuantil (0,340 y 0,728), con desviaciones
entre particiones de 0,004 a 0,020. Se esperaba que el escalador robusto ayudara por las colas largas,
pero ocurrió lo contrario. Una explicación plausible es que en variables como `previous` y `pdays` el
rango intercuartílico es cero, de modo que el escalador no las comprime y valores como 275 quedan
sin reducir, justo en los pocos clientes contactados antes, y el kernel RBF los trata como
distantes de todo. Usé `StandardScaler`.

## Guardado de las particiones

Guardé las particiones **sin transformar**, junto con la lista de columnas y el escalador elegido. Cada
notebook posterior construye sus propios `Pipeline` y los ajusta solo con train, de modo que ninguna
transformación se calcula con datos de prueba.

In [10]:
import joblib

paquete = {
    "X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test,
    "cols_num": cols_num, "cols_cat": cols_cat, "escalador_svm": escalador_elegido,
}
joblib.dump(paquete, os.path.join(RUTA, "datos_preparados.joblib"))

resumen = {
    "filas": len(df), "train": len(X_train), "test": len(X_test),
    "tasa_train": float(y_train.mean()), "tasa_test": float(y_test.mean()),
    "escalador": escalador_elegido, "duplicados_sin_duration": duplicados,
}
with open(os.path.join(RUTA, "resumen_nb03.json"), "w") as f:
    json.dump(resumen, f)
print("Particiones guardadas en:", RUTA)

Particiones guardadas en: C:\Users\USUARIO\Documents\6_SEMESTRE\MACHINE LEARNING\Bank_Marketing\bank_marketing
